In [10]:
import sys

sys.path.append("..\..")
from machex.dataset_class.dataset import MaCheXDataset, ChestXrayDataset
#from evaluation.utils.class_peft_utils import *
from torch import nn
from torch import optim
from tqdm import tqdm
from matplotlib import pyplot as plt
import torch 
import numpy as np
import torchxrayvision as xrv
from sklearn.metrics import f1_score, classification_report, confusion_matrix,precision_score, recall_score
import torch.nn.functional as F

In [11]:
# config

config = {

    'data': {
        'Machex_path': '..\..\machex\machex_pediatric_classifier_5',
    },
    'model': {
        'model_name': 'densenet121-res224-all',
        'fine_tuning_mode': 'head_only', # 'head_only' or 'full'
        'targets': [
            "No finding", "Bronchitis", "Brocho-pneumonia", "Bronchiolitis", "Pneumonia"
            ]
    },

    'test': {
        'lr_head': 1e-3,
        'lr_backbone': 1e-5,
        'epochs': 20,
        'batch_size': 32, 
        
    }

}

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [12]:
def get_model(config, device):

    print(f"Initializing model in mode: {config['fine_tuning_mode']}")
    model = xrv.models.DenseNet(weights=config['model_name'])
    
    num_features = model.classifier.in_features
    new_head = nn.Sequential(
        nn.Linear(num_features, len(config['targets'])),
    )

    if config['fine_tuning_mode'] == 'head_only':
        for param in model.parameters():
            param.requires_grad = False
        model.classifier = new_head
        
    elif config['fine_tuning_mode'] == 'full':
        model.classifier = new_head
    
    return model.to(device)

In [13]:
dataset_test  = MaCheXDataset(config['data']['Machex_path'], split="test")

test_loader = torch.utils.data.DataLoader(
    dataset_test, 
    batch_size=config['test']['batch_size'], 
    shuffle=False,
    num_workers=1
)

model = get_model(config['model'], device)
model_path = f"best_model_{config['model']['fine_tuning_mode']}_5.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
print(f"Loaded best model weights from {model_path}")

criterion = nn.BCEWithLogitsLoss()

Initializing model in mode: head_only
Loaded best model weights from best_model_head_only_5.pth


In [14]:
def forward_new_head(model, inputs):
    features = model.features(inputs)
    out = F.relu(features, inplace=True)
    out = F.adaptive_avg_pool2d(out, (1, 1))
    out = torch.flatten(out, 1)
    outputs = model.classifier(out) 
    return outputs

def test_model(model, loader, criterion, device):
    """Evaluates the model on the test set and calculates Precision, Recall, and F1."""
    model.eval()
    y_true, y_pred = [], []
    total_loss = 0

    pbar = tqdm(loader, desc="Testing Model", unit="batch")
    
    with torch.no_grad():
        for batch in pbar:
            inputs = batch['img'].to(device)
            labels = batch['label_tensor'].to(device) 

            outputs = forward_new_head(model, inputs)
            preds = torch.sigmoid(outputs)

            loss = criterion(outputs, labels.float())
            total_loss += loss.item()

            y_true.append(labels.cpu().numpy())
            y_pred.append(preds.cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    
    # Binarize predictions (threshold 0.5)
    y_pred_bin = (y_pred > 0.5).astype(int)
    
    # Calculate Macro Metrics
    macro_precision = precision_score(y_true, y_pred_bin, average='macro', zero_division=0)
    macro_recall = recall_score(y_true, y_pred_bin, average='macro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred_bin, average='macro', zero_division=0)
    
    avg_loss = total_loss / len(loader)
    
    # Generate a detailed per-class report
    target_names = config['model']['targets']
    report = classification_report(y_true, y_pred_bin, target_names=target_names, zero_division=0)
    
    return avg_loss, macro_precision, macro_recall, macro_f1, report

# 3. Run the evaluation
print("Starting Evaluation on Test Set...")
test_loss, precision, recall, f1, class_report = test_model(model, test_loader, criterion, device)

print("\n" + "="*50)
print("FINAL TEST RESULTS")
print("="*50)
print(f"Test Loss:      {test_loss:.4f}")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall:    {recall:.4f}")
print(f"Macro F1-Score:  {f1:.4f}")
print("\nPer-Class Report:")
print(class_report)

Starting Evaluation on Test Set...


Testing Model: 100%|██████████| 25/25 [00:52<00:00,  2.10s/batch]


FINAL TEST RESULTS
Test Loss:      0.5446
Macro Precision: 0.2938
Macro Recall:    0.6838
Macro F1-Score:  0.3658

Per-Class Report:
                  precision    recall  f1-score   support

      No finding       0.82      0.81      0.82       558
      Bronchitis       0.21      0.79      0.33       126
Brocho-pneumonia       0.13      0.87      0.22        39
   Bronchiolitis       0.12      0.46      0.19        37
       Pneumonia       0.20      0.49      0.28        35

       micro avg       0.40      0.78      0.53       795
       macro avg       0.29      0.68      0.37       795
    weighted avg       0.63      0.78      0.66       795
     samples avg       0.52      0.78      0.60       795

